# Route Legibility Baseline (Stage 1)

This notebook tests the core Phase 1 hypothesis: **B-KQ is close but reads as far** -
the city-core and transport-hub approaches to the Ryder Street gateway are
perceptually and spatially hard to follow.

For each inbound route family it computes a transparent **Route Legibility Index (RLI)**
from five separately visible components (directness, turn burden, intersection
complexity, crossing/major-road exposure, continuity), identifies decision points, and
visualises where the journey breaks down.

This runs on the Gate 0B audited OSM walk network (Evidence Level 3). All outputs are
**descriptive only**. No diagnostic, recommendation, or impact claim is made: the RLI is
a structured hypothesis, the thresholds are documented assumptions, and the anchors are
provisional and not field-validated.

In [ ]:
from __future__ import annotations
from pathlib import Path
import sys, ast, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import networkx as nx
import osmnx as ox
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
PHASE1_ROOT = PROJECT_ROOT.parent if PROJECT_ROOT.name == "notebooks" else PROJECT_ROOT / "phase1_spinelens_ai"
sys.path.insert(0, str(PHASE1_ROOT / "src"))

from spinelens.spatial import audit
from spinelens.metrics import legibility as lg

DATA = PHASE1_ROOT / "data"
FIG_DIR = PHASE1_ROOT / "outputs" / "reports" / "route_legibility_media"
TABLES = PHASE1_ROOT / "outputs" / "tables"
REPORTS = PHASE1_ROOT / "outputs" / "reports"
for d in (FIG_DIR, TABLES, REPORTS):
    d.mkdir(parents=True, exist_ok=True)

WALK_SPEED_MS = 1.33  # ~4.8 km/h standard adult walking speed

# Audited walk network (Gate 0B, Evidence Level 3).
G = ox.load_graphml(DATA / "raw" / "osm_network" / "gate0b_osm_walk_graph.graphml")
und = G.to_undirected()
largest = max(nx.connected_components(und), key=len)
coords = {n: (float(d["y"]), float(d["x"])) for n, d in G.nodes(data=True)}
coords_largest = {n: c for n, c in coords.items() if n in largest}

def street_count(n):
    v = G.nodes[n].get("street_count")
    try:
        return int(float(v))
    except (TypeError, ValueError):
        return und.degree(n)

def _hw_list(hw):
    if isinstance(hw, list):
        return hw
    if isinstance(hw, str) and hw.startswith("["):
        try:
            return ast.literal_eval(hw)
        except (ValueError, SyntaxError):
            return [hw]
    return [hw]

# Precompute per-node major-road exposure severity from incident edges.
node_severity = {n: 0.0 for n in G.nodes}
for u, v, d in G.edges(data=True):
    sev = max((lg.crossing_severity(h) for h in _hw_list(d.get("highway"))), default=0.0)
    if sev > 0:
        node_severity[u] = max(node_severity[u], sev)
        node_severity[v] = max(node_severity[v], sev)

nodes = pd.read_csv(DATA / "route_nodes_phase1.csv").set_index("node_id")
families = pd.read_csv(DATA / "route_families_phase1.csv")
inbound = families[families["origin_node_id"] != families["gateway_node_id"]].copy()
print(f"graph nodes={G.number_of_nodes()} largest_comp={len(largest)} | inbound route families={len(inbound)}")

## Route candidates and Route Legibility Index

In [ ]:
def snap(node_id):
    pt = (float(nodes.loc[node_id, "latitude"]), float(nodes.loc[node_id, "longitude"]))
    nid, dist = audit.nearest_node(coords_largest, pt)
    return nid, dist

routes = {}     # family_id -> list of graph nodes
records = []
for _, fam in inbound.iterrows():
    fid = fam["route_family_id"]
    o_node, o_snap = snap(fam["origin_node_id"])
    g_node, g_snap = snap(fam["gateway_node_id"])
    path = nx.shortest_path(G, o_node, g_node, weight="length")
    routes[fid] = path
    network_m = nx.shortest_path_length(G, o_node, g_node, weight="length")
    pcoords = [coords[n] for n in path]
    straight_m = audit.haversine_m(coords[o_node], coords[g_node])
    route_km = network_m / 1000.0

    interior = path[1:-1]
    decision_nodes = [n for n in interior if street_count(n) >= 3]
    severity_sum = sum(node_severity[n] for n in interior)
    total_turn = lg.total_turning_deg(pcoords)

    comps = {
        "directness": lg.directness_score(straight_m, network_m),
        "turn_burden": lg.turn_burden_score(total_turn, route_km),
        "intersection_complexity": lg.intersection_complexity_score(len(decision_nodes), route_km),
        "crossing_burden": lg.crossing_burden_score(severity_sum, route_km),
        "continuity": lg.continuity_score(pcoords, route_km),
    }
    rli = lg.weighted_legibility_score(comps)
    records.append({
        "route_family": fid,
        "origin": fam["origin_node_id"],
        "priority": fam["priority"],
        "distance_m": round(network_m, 0),
        "straight_m": round(straight_m, 0),
        "walk_time_min": round(network_m / WALK_SPEED_MS / 60, 1),
        "turns_deg": round(total_turn, 0),
        "decision_points": len(decision_nodes),
        "major_road_exposure": round(severity_sum, 1),
        "origin_snap_m": round(o_snap, 1),
        **{k: round(v, 3) for k, v in comps.items()},
        "RLI": round(rli, 3),
    })

results = pd.DataFrame(records).sort_values("RLI", ascending=False).reset_index(drop=True)
display(results)

## Comparison matrix

In [ ]:
cols = ["route_family", "priority", "distance_m", "walk_time_min", "directness",
        "turn_burden", "intersection_complexity", "crossing_burden", "continuity", "RLI"]
matrix = results[cols].copy()
matrix.to_csv(TABLES / "route_legibility_comparison.csv", index=False)
print("saved:", (TABLES / "route_legibility_comparison.csv").relative_to(PHASE1_ROOT))
display(matrix)
print(f"Most legible: {results.iloc[0]['route_family']} (RLI {results.iloc[0]['RLI']})")
print(f"Least legible: {results.iloc[-1]['route_family']} (RLI {results.iloc[-1]['RLI']})")

## Decision points / confusion nodes

Nodes used by the inbound routes, ranked by junction complexity and how many routes pass through them (corridor convergence).

In [ ]:
from collections import Counter
node_use = Counter()
for path in routes.values():
    for n in path[1:-1]:
        node_use[n] += 1

dp = []
for n, used in node_use.items():
    sc = street_count(n)
    if sc >= 3:
        dp.append({"node": n, "lat": coords[n][0], "lon": coords[n][1],
                   "street_count": sc, "routes_using": used,
                   "major_road_severity": node_severity[n]})
dp_df = pd.DataFrame(dp).sort_values(["routes_using", "street_count", "major_road_severity"],
                                     ascending=False).reset_index(drop=True)
dp_df.to_csv(TABLES / "route_decision_points.csv", index=False)
print(f"decision nodes on inbound routes: {len(dp_df)} | shared by >=2 routes: {int((dp_df['routes_using']>=2).sum())}")
display(dp_df.head(12))

## Visuals

In [ ]:
import matplotlib.cm as cm
edges_4326 = ox.graph_to_gdfs(G, nodes=False).to_crs(4326)
palette = {fid: cm.tab10(i) for i, fid in enumerate(routes)}

# FIGURE A - inbound routes
fig, ax = plt.subplots(figsize=(11, 9))
edges_4326.plot(ax=ax, color="#e0e0e0", linewidth=0.4, zorder=1)
for fid, path in routes.items():
    lat = [coords[n][0] for n in path]; lon = [coords[n][1] for n in path]
    ax.plot(lon, lat, color=palette[fid], linewidth=2.4, zorder=3, label=fid)
    ax.scatter(lon[0], lat[0], color=palette[fid], s=70, edgecolor="white", zorder=4)
g_node, _ = snap("ryder_street_pavilion_search_area")
ax.scatter(coords[g_node][1], coords[g_node][0], color="black", marker="*", s=320,
           edgecolor="white", zorder=6, label="ryder gateway")
ax.set_aspect(1/np.cos(np.radians(coords[g_node][0])))
ax.legend(fontsize=7, loc="upper left"); ax.set_title("Inbound route candidates to the Ryder Street gateway")
fig.savefig(FIG_DIR / "figA_routes.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
# FIGURE B - RLI and component contributions
comp_cols = ["directness", "turn_burden", "intersection_complexity", "crossing_burden", "continuity"]
weights = lg.DEFAULT_WEIGHTS
wmap = {"directness": weights.directness, "turn_burden": weights.turn_burden,
        "intersection_complexity": weights.intersection_complexity,
        "crossing_burden": weights.crossing_burden, "continuity": weights.continuity}
contrib = results.set_index("route_family")[comp_cols].mul(pd.Series(wmap))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].barh(results["route_family"], results["RLI"], color="#1a73e8")
axes[0].set_xlim(0, 1); axes[0].set_title("Route Legibility Index (1 = most legible)")
axes[0].invert_yaxis()
bottom = np.zeros(len(contrib))
for c in comp_cols:
    axes[1].barh(contrib.index, contrib[c], left=bottom, label=c)
    bottom += contrib[c].values
axes[1].set_title("Weighted component contributions to RLI"); axes[1].legend(fontsize=7)
axes[1].invert_yaxis()
fig.tight_layout(); fig.savefig(FIG_DIR / "figB_rli_components.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
# FIGURE C - decision points (size = junction complexity, colour = routes sharing)
fig, ax = plt.subplots(figsize=(11, 9))
edges_4326.plot(ax=ax, color="#e8e8e8", linewidth=0.4, zorder=1)
for fid, path in routes.items():
    lat = [coords[n][0] for n in path]; lon = [coords[n][1] for n in path]
    ax.plot(lon, lat, color="#bdbdbd", linewidth=1.2, zorder=2)
sc_plot = ax.scatter(dp_df["lon"], dp_df["lat"], s=dp_df["street_count"]*22,
                     c=dp_df["routes_using"], cmap="YlOrRd", edgecolor="black",
                     linewidth=0.4, zorder=4)
plt.colorbar(sc_plot, ax=ax, label="routes sharing the node")
ax.set_aspect(1/np.cos(np.radians(dp_df["lat"].mean())))
ax.set_title("Decision points: junction complexity and corridor convergence")
fig.savefig(FIG_DIR / "figC_decision_points.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
# FIGURE D - component heatmap (routes x components)
fig, ax = plt.subplots(figsize=(9, 4.8))
hm = results.set_index("route_family")[comp_cols]
im = ax.imshow(hm.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(comp_cols))); ax.set_xticklabels(comp_cols, rotation=30, ha="right")
ax.set_yticks(range(len(hm))); ax.set_yticklabels(hm.index)
for i in range(len(hm)):
    for j in range(len(comp_cols)):
        ax.text(j, i, f"{hm.values[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(im, ax=ax, label="component score (1 = most legible)")
ax.set_title("Legibility component profile by route")
fig.tight_layout(); fig.savefig(FIG_DIR / "figD_component_heatmap.png", dpi=130, bbox_inches="tight"); plt.show()

## Findings note

In [ ]:
best = results.iloc[0]; worst = results.iloc[-1]
shared = int((dp_df["routes_using"] >= 2).sum())
lines = [
    "# Route Legibility Baseline Note (Stage 1)",
    "",
    f"Generated from the Gate 0B audited walk network (Evidence Level 3). Descriptive only.",
    "",
    "## Method",
    "",
    "Route Legibility Index (RLI) = weighted sum of five normalised (0..1) components:",
    f"directness {lg.DEFAULT_WEIGHTS.directness}, turn_burden {lg.DEFAULT_WEIGHTS.turn_burden}, "
    f"intersection_complexity {lg.DEFAULT_WEIGHTS.intersection_complexity}, "
    f"crossing_burden {lg.DEFAULT_WEIGHTS.crossing_burden}, continuity {lg.DEFAULT_WEIGHTS.continuity}.",
    "Higher RLI = easier to read. Thresholds are documented assumptions in metrics/legibility.py.",
    "",
    "## Inbound route results",
    "",
    "| Route | Walk min | Directness | RLI |",
    "|---|---:|---:|---:|",
]
for _, r in results.iterrows():
    lines.append(f"| {r['route_family']} | {r['walk_time_min']} | {r['directness']} | {r['RLI']} |")
lines += [
    "",
    "## Findings (descriptive)",
    "",
    f"- Most legible inbound approach: {best['route_family']} (RLI {best['RLI']}).",
    f"- Least legible inbound approach: {worst['route_family']} (RLI {worst['RLI']}).",
    f"- Decision nodes on inbound routes: {len(dp_df)}; shared by 2+ routes (corridor convergence): {shared}.",
    f"- Mean inbound walk time: {round(results['walk_time_min'].mean(),1)} min over {round(results['distance_m'].mean()/1000,2)} km average.",
    "",
    "## Caveats",
    "",
    "- RLI is a structured hypothesis, not a measured outcome; weights/thresholds need sensitivity testing.",
    "- Crossing burden is a major-road exposure proxy from OSM highway tags, not a verified crossing count.",
    "- Landmark/active-frontage support is not yet included (needs POI/frontage data).",
    "- Anchors are provisional and not field-validated; OSM is volunteered data pending OS cross-check.",
    "",
    "## Next steps",
    "",
    "- Sensitivity analysis on RLI weights and thresholds.",
    "- Cross-check crossing burden against OS Open Roads classification (Evidence Level 4).",
    "- Use shared decision nodes to seed tactical-corridor and wayfinder-placement experiments.",
]
note = "\n".join(lines)
(REPORTS / "route_legibility_baseline_note.md").write_text(note, encoding="utf-8")
print(note)

## What this unlocks

A descriptive, component-wise legibility baseline for all five inbound approaches, plus a
ranked decision-point set. This is the evidence base for the next experiments: weight
sensitivity, tactical-corridor synthesis (shared segments), and budget-aware wayfinder
placement (clarity per pound). No funding-facing claim is made here.

# Sensitivity analysis

The Route Legibility Index depends on two sets of assumptions: the **component weights**
and the **reference thresholds**. Before trusting the ranking, we test whether it is
robust to those choices or merely an artefact of them. Three tests:

1. **Monte Carlo over weights** - 4000 random weightings (Dirichlet); how stable is each
   route's rank?
2. **Named weighting schemes** - equal, directness-led, crossing-led, complexity-led, default;
   do the rankings agree (Spearman)?
3. **Threshold tornado** - vary each reference band +/-50%; how far does each route's RLI move?

All reproducible (fixed seed). Still descriptive only.

In [ ]:
# Reusable inputs (all in scope from the baseline above).
comp_cols = list(lg.COMPONENT_KEYS)
comp_matrix = results.set_index("route_family")[comp_cols]
order = results["route_family"].tolist()  # baseline RLI order (best -> worst)
clamp01 = lambda v: max(0.0, min(1.0, v))

# Exact raw per-km inputs (recomputed from the stored route paths).
default_refs = {
    "turn": lg.TURN_REF_DEG_PER_KM, "decision": lg.DECISION_REF_PER_KM,
    "crossing": lg.CROSSING_REF_SEVERITY_PER_KM, "continuity": lg.CONTINUITY_REF_TURNS_PER_KM,
}
rawmap = {}
res_idx = results.set_index("route_family")
for fid, path in routes.items():
    pc = [coords[n] for n in path]
    km = res_idx.loc[fid, "distance_m"] / 1000.0
    interior = path[1:-1]
    rawmap[fid] = {
        "directness_ratio": float(res_idx.loc[fid, "directness"]),
        "turn_per_km": lg.total_turning_deg(pc) / km,
        "decision_per_km": sum(1 for n in interior if street_count(n) >= 3) / km,
        "sig_per_km": sum(1 for a in lg.turn_angles(pc) if a >= lg.SIGNIFICANT_TURN_DEG) / km,
        "severity_per_km": sum(node_severity[n] for n in interior) / km,
    }

def components_from_raw(r, refs):
    return {
        "directness": clamp01(r["directness_ratio"]),
        "turn_burden": clamp01(1 - r["turn_per_km"] / refs["turn"]),
        "intersection_complexity": clamp01(1 - r["decision_per_km"] / refs["decision"]),
        "crossing_burden": clamp01(1 - r["severity_per_km"] / refs["crossing"]),
        "continuity": clamp01(1 - r["sig_per_km"] / refs["continuity"]),
    }

print("raw per-km inputs:")
display(pd.DataFrame(rawmap).T.round(2))

## 1. Monte Carlo over weights (rank stability)

In [ ]:
rng = np.random.default_rng(42)
N = 4000
W = rng.dirichlet(np.ones(len(comp_cols)), size=N)          # N x 5 weights
rli_samples = comp_matrix.values @ W.T                       # routes x N
rli_df = pd.DataFrame(rli_samples.T, columns=comp_matrix.index)

mc_stats = pd.DataFrame({
    "rli_mean": rli_df.mean(), "rli_std": rli_df.std(),
    "rli_min": rli_df.min(), "rli_max": rli_df.max(),
}).loc[order].round(3)
display(mc_stats)

ranks = rli_df.rank(axis=1, ascending=False, method="min").astype(int)
freq = pd.DataFrame(0.0, index=order, columns=range(1, len(order) + 1))
for r in order:
    for k, v in ranks[r].value_counts(normalize=True).items():
        freq.loc[r, k] = v
print("P(rank) per route:")
display(freq.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
# RLI distribution under random weights
axes[0].boxplot([rli_df[r] for r in order], vert=False, labels=order)
axes[0].invert_yaxis(); axes[0].set_xlim(0, 1)
axes[0].set_title("RLI distribution over 4000 random weightings")
axes[0].set_xlabel("Route Legibility Index")
# Rank-frequency heatmap
im = axes[1].imshow(freq.values, cmap="Blues", vmin=0, vmax=1, aspect="auto")
axes[1].set_xticks(range(len(order))); axes[1].set_xticklabels(range(1, len(order) + 1))
axes[1].set_yticks(range(len(order))); axes[1].set_yticklabels(order)
axes[1].set_xlabel("rank (1 = most legible)")
for i in range(len(order)):
    for j in range(len(order)):
        axes[1].text(j, i, f"{freq.values[i, j]:.2f}", ha="center", va="center", fontsize=8)
axes[1].set_title("P(rank) under random weightings")
plt.colorbar(im, ax=axes[1], label="probability")
fig.tight_layout(); fig.savefig(FIG_DIR / "figE_weight_montecarlo.png", dpi=130, bbox_inches="tight"); plt.show()

## 2. Named weighting schemes

In [ ]:
w = lg.DEFAULT_WEIGHTS
schemes = {
    "equal":            {k: 1 for k in comp_cols},
    "default":          {"directness": w.directness, "turn_burden": w.turn_burden,
                         "intersection_complexity": w.intersection_complexity,
                         "crossing_burden": w.crossing_burden, "continuity": w.continuity},
    "directness_led":   {"directness": 0.50, "turn_burden": 0.125, "intersection_complexity": 0.125,
                         "crossing_burden": 0.125, "continuity": 0.125},
    "crossing_led":     {"directness": 0.15, "turn_burden": 0.15, "intersection_complexity": 0.15,
                         "crossing_burden": 0.40, "continuity": 0.15},
    "complexity_led":   {"directness": 0.15, "turn_burden": 0.225, "intersection_complexity": 0.275,
                         "crossing_burden": 0.125, "continuity": 0.225},
}
scheme_rli = pd.DataFrame(
    {name: [lg.score_with_weights(comp_matrix.loc[r].to_dict(), wts) for r in comp_matrix.index]
     for name, wts in schemes.items()},
    index=comp_matrix.index,
).loc[order].round(3)
scheme_ranks = scheme_rli.rank(ascending=False, method="min").astype(int)
print("RLI by scheme:"); display(scheme_rli)
print("Rank by scheme (1 = most legible):"); display(scheme_ranks)
spearman = scheme_rli.corr(method="spearman")
print("Spearman rank correlation between schemes:"); display(spearman.round(3))

## 3. Threshold tornado (+/-50%)

In [ ]:
def rli_under(refs, weights):
    return {fid: lg.score_with_weights(components_from_raw(rawmap[fid], refs), weights) for fid in order}

base_w = schemes["default"]
baseline = rli_under(default_refs, base_w)
lo, hi = {f: baseline[f] for f in order}.copy(), {f: baseline[f] for f in order}.copy()
for param in default_refs:
    for factor in (0.5, 1.5):
        refs = dict(default_refs); refs[param] = default_refs[param] * factor
        r = rli_under(refs, base_w)
        for f in order:
            lo[f] = min(lo[f], r[f]); hi[f] = max(hi[f], r[f])

fig, ax = plt.subplots(figsize=(10, 5))
y = range(len(order))
ax.hlines(list(y), [lo[f] for f in order], [hi[f] for f in order], color="#9aa0a6", linewidth=6, alpha=0.6)
ax.scatter([baseline[f] for f in order], list(y), color="#1a73e8", zorder=3, label="baseline RLI")
ax.set_yticks(list(y)); ax.set_yticklabels(order); ax.invert_yaxis(); ax.set_xlim(0, 1)
ax.set_xlabel("RLI range under +/-50% threshold perturbation"); ax.legend()
ax.set_title("Threshold sensitivity of each route's RLI")
fig.tight_layout(); fig.savefig(FIG_DIR / "figF_threshold_tornado.png", dpi=130, bbox_inches="tight"); plt.show()

tornado = pd.DataFrame({"baseline": baseline, "min": lo, "max": hi}).loc[order]
tornado["range"] = (tornado["max"] - tornado["min"]).round(3)
display(tornado.round(3))

## Robustness summary

In [ ]:
top2 = set(order[:2]); bottom2 = set(order[-2:])
p_top2 = {r: float((ranks[r] <= 2).mean()) for r in order}
p_bottom2 = {r: float((ranks[r] >= len(order) - 1).mean()) for r in order}
# does the most/least legible ever change across named schemes?
best_per_scheme = {s: scheme_rli[s].idxmax() for s in schemes}
worst_per_scheme = {s: scheme_rli[s].idxmin() for s in schemes}
stable_best = len(set(best_per_scheme.values())) == 1
stable_worst = len(set(worst_per_scheme.values())) == 1
most_sensitive = tornado["range"].idxmax()

lines = [
    "# Route Legibility Sensitivity Note (Stage 1.5)",
    "",
    "Reproducible (seed 42). Descriptive only. Tests robustness of the RLI ranking to",
    "weight and threshold assumptions.",
    "",
    "## Weight Monte Carlo (4000 Dirichlet samples)",
    "",
    "| Route | P(top-2) | P(bottom-2) | RLI mean | RLI min-max |",
    "|---|---:|---:|---:|---|",
]
for r in order:
    lines.append(f"| {r} | {p_top2[r]:.2f} | {p_bottom2[r]:.2f} | {mc_stats.loc[r,'rli_mean']:.3f} | "
                 f"{mc_stats.loc[r,'rli_min']:.3f}-{mc_stats.loc[r,'rli_max']:.3f} |")
lines += [
    "",
    "## Named schemes",
    "",
    f"- Most legible route is stable across all schemes: {stable_best} "
    f"({sorted(set(best_per_scheme.values()))}).",
    f"- Least legible route is stable across all schemes: {stable_worst} "
    f"({sorted(set(worst_per_scheme.values()))}).",
    f"- Minimum pairwise Spearman correlation between schemes: {spearman.values[spearman.values < 1].min():.3f}.",
    "",
    "## Threshold tornado",
    "",
    f"- Most threshold-sensitive route: {most_sensitive} (RLI range {tornado.loc[most_sensitive,'range']:.3f}).",
    f"- Maximum RLI range across all routes under +/-50% thresholds: {tornado['range'].max():.3f}.",
    "",
    "## Robust conclusions",
    "",
    f"- Top tier (high P(top-2)): {', '.join(r for r in order if p_top2[r] >= 0.6) or 'none'}.",
    f"- Bottom tier (high P(bottom-2)): {', '.join(r for r in order if p_bottom2[r] >= 0.6) or 'none'}.",
    "- Conclusions that survive the assumptions can inform corridor/wayfinder priorities;",
    "  conclusions that flip under reweighting must be reported as uncertain.",
    "",
    "## Caveat",
    "",
    "Robustness to assumptions is not validity. The components are still proxies on volunteered",
    "data with provisional anchors; field validation is required before funding-facing claims.",
]
note = "\n".join(lines)
(REPORTS / "route_legibility_sensitivity_note.md").write_text(note, encoding="utf-8")
print(note)